# Age–Sex-Adjusted Municipality Benchmarking

**Author:** Mahdi Dadgar  
**Input:** `data/processed/vektis_2023_clean.parquet`

## Why this analysis matters

Municipalities differ substantially in their population structure. A municipality with more older residents may have higher healthcare costs even when its care-use pattern is similar to the national pattern.

For that reason, this notebook does not compare raw costs alone. It applies **indirect age–sex standardisation** to estimate how much each municipality would be expected to spend if its residents experienced the national average healthcare cost rate for their age and sex group.

## Standardised Cost Ratio

For each municipality:

> **Standardised Cost Ratio (SCR) = actual healthcare cost / expected healthcare cost**

Interpretation:

- **SCR above 1.00:** actual costs are higher than the age–sex-adjusted expectation
- **SCR close to 1.00:** actual costs are close to the expectation
- **SCR below 1.00:** actual costs are lower than the expectation

An SCR is a descriptive benchmark, not a judgement of efficiency or quality. Differences may also reflect morbidity, socioeconomic conditions, regional supply, coding practices, patient preferences, or other factors that are not available in this dataset.

## Notebook outputs

This notebook produces:

- municipality-level actual and expected costs
- SCR and euro-gap measures
- a transparent population-size caution flag
- a sensitivity comparison of rankings
- an 11-category breakdown of actual versus expected costs


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

current_path = Path.cwd()
PROJECT_ROOT = current_path.parent if current_path.name == "notebooks" else current_path

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"

processed_path = DATA_PROCESSED / "vektis_2023_clean.parquet"

if not processed_path.exists():
    raise FileNotFoundError(
        "The cleaned Parquet file was not found. Run Notebook 01 first."
    )

df = pd.read_parquet(processed_path)

required_columns = {
    "gemeentenaam",
    "leeftijdsklasse",
    "geslacht",
    "aantal_verzekerdejaren",
    "total_healthcare_cost",
}

missing_required_columns = required_columns.difference(df.columns)

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_required_columns)}"
    )

cost_cols = [
    column
    for column in df.columns
    if column.startswith("kosten_")
]

print("Loaded cleaned dataset:", df.shape)
print("Municipalities:", df["gemeentenaam"].nunique())
print("Healthcare cost columns:", len(cost_cols))
display(df.head())

Loaded cleaned dataset: (12988, 34)
Municipalities: 342
Healthcare cost columns: 26


,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,kosten_huisarts_overig,kosten_hulpmiddelen,kosten_mondzorg,kosten_paramedische_zorg_fysioth,kosten_paramedische_zorg_overig,kosten_ziekenvervoer_zittend,kosten_ziekenvervoer_liggend,kosten_kraamzorg,kosten_verloskundige_zorg,kosten_grensoverschrijdende_zorg,kosten_eerstelijns_ondersteuning,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz,total_healthcare_cost,cost_per_insured_year,specialist_care_cost_per_insured_year
0,V,5 t/m 9 jaar,AMERSFOORT,4361,4331.92,1993077.92,321453.33,324320.16,112567.49,90834.26,330502.14,133320.18,802215.31,95713.96,624176.93,6676.52,13046.81,0.0,0.00,5701.64,0,0.0,0.0,308714.33,0.0,0.0,0.0,222796.62,0.0,0.0,0.0,5385117.60,1243.124896,460.091119
1,V,10 t/m 14 jaar,OOST GELRE,749,742.98,212622.21,36365.89,55490.34,17453.58,9359.63,40696.29,34310.67,189190.36,41203.68,25046.79,0.00,3189.04,0.0,0.00,1099.43,0,0.0,0.0,0.00,0.0,0.0,0.0,9088.73,0.0,0.0,0.0,675116.64,908.660583,286.174877
2,V,10 t/m 14 jaar,OOSTERHOUT,1498,1487.53,598245.35,82667.47,113762.71,39852.67,25524.37,60438.20,53450.85,309260.83,88869.47,26446.71,0.00,6962.66,0.0,0.00,2806.82,0,0.0,0.0,6577.83,0.0,0.0,0.0,27837.98,0.0,0.0,0.0,1442703.92,969.865428,402.173637
3,V,10 t/m 14 jaar,OOSTSTELLINGWERF,647,643.94,296500.62,38004.44,48648.18,16924.25,13611.31,35866.44,38416.84,98746.00,32716.34,12593.34,322.86,955.86,0.0,433.79,4840.87,0,0.0,0.0,1306.40,0.0,0.0,0.0,6293.08,0.0,0.0,0.0,646180.62,1003.479548,460.447588
4,V,10 t/m 14 jaar,OOSTZAAN,245,244.07,63902.89,8518.97,18259.37,6162.90,4042.80,17090.64,842.54,66513.11,12291.22,9157.45,0.00,878.58,0.0,0.00,113.86,0,0.0,0.0,0.00,0.0,0.0,0.0,3284.54,0.0,0.0,0.0,211058.87,864.747286,261.821977


## 1. Build national age–sex reference rates

For every age–sex group, the national reference rate is calculated as:

> total healthcare cost in the group / total insured years in the group

The reference rates are weighted by insured years, not by municipality count.

In [2]:
age_sex_reference = (
    df.groupby(
        ["geslacht", "leeftijdsklasse"],
        observed=True,
    )
    .agg(
        national_actual_cost=(
            "total_healthcare_cost",
            "sum",
        ),
        national_insured_years=(
            "aantal_verzekerdejaren",
            "sum",
        ),
    )
    .reset_index()
)

age_sex_reference["national_rate_per_insured_year"] = (
    age_sex_reference["national_actual_cost"]
    / age_sex_reference["national_insured_years"]
)

if (
    age_sex_reference["national_insured_years"]
    <= 0
).any():
    raise ValueError(
        "At least one national age-sex cell has zero or negative insured years."
    )

def age_sort_value(age_label):
    age_label = str(age_label)
    if age_label == "90+":
        return 90
    return int(age_label.split()[0])

age_sex_reference["age_sort"] = (
    age_sex_reference["leeftijdsklasse"]
    .map(age_sort_value)
)

age_sex_reference = (
    age_sex_reference
    .sort_values(
        ["age_sort", "geslacht"],
    )
    .reset_index(drop=True)
)

reference_rate_pivot = (
    age_sex_reference
    .pivot(
        index="leeftijdsklasse",
        columns="geslacht",
        values="national_rate_per_insured_year",
    )
    .reindex(
        age_sex_reference[
            ["leeftijdsklasse", "age_sort"]
        ]
        .drop_duplicates()
        .sort_values("age_sort")[
            "leeftijdsklasse"
        ]
    )
)

print("National healthcare cost rate per insured year:")
display(reference_rate_pivot.round(2))

National healthcare cost rate per insured year:


geslacht,M,V
leeftijdsklasse,,
0 t/m 4 jaar,2710.46,2192.12
5 t/m 9 jaar,1527.35,1259.88
10 t/m 14 jaar,1286.43,1157.73
15 t/m 19 jaar,1326.76,1560.15
20 t/m 24 jaar,1316.82,2009.71
25 t/m 29 jaar,1387.62,2561.67
30 t/m 34 jaar,1480.46,3107.96
35 t/m 39 jaar,1555.26,2732.07
40 t/m 44 jaar,1715.50,2382.73


## 2. Calculate expected cost for each municipality

Each municipality–age–sex row is assigned the corresponding national reference rate.

Expected cost is then calculated as:

> municipality insured years × national age–sex reference rate

Summing these expected values across all age–sex groups gives the municipality's total expected cost.

In [3]:
df_with_rates = df.merge(
    age_sex_reference[
        [
            "geslacht",
            "leeftijdsklasse",
            "national_rate_per_insured_year",
        ]
    ],
    on=["geslacht", "leeftijdsklasse"],
    how="left",
    validate="many_to_one",
)

missing_reference_rates = (
    df_with_rates[
        "national_rate_per_insured_year"
    ]
    .isna()
    .sum()
)

if missing_reference_rates:
    raise ValueError(
        f"{missing_reference_rates} rows did not match a national reference rate."
    )

df_with_rates["expected_cost"] = (
    df_with_rates["aantal_verzekerdejaren"]
    * df_with_rates["national_rate_per_insured_year"]
)

display(
    df_with_rates[
        [
            "gemeentenaam",
            "leeftijdsklasse",
            "geslacht",
            "aantal_verzekerdejaren",
            "total_healthcare_cost",
            "national_rate_per_insured_year",
            "expected_cost",
        ]
    ].head()
)

,gemeentenaam,leeftijdsklasse,geslacht,aantal_verzekerdejaren,total_healthcare_cost,national_rate_per_insured_year,expected_cost
0,AMERSFOORT,5 t/m 9 jaar,V,4331.92,5385117.60,1259.882856,5.457712e+06
1,OOST GELRE,10 t/m 14 jaar,V,742.98,675116.64,1157.725563,8.601669e+05
2,OOSTERHOUT,10 t/m 14 jaar,V,1487.53,1442703.92,1157.725563,1.722152e+06
3,OOSTSTELLINGWERF,10 t/m 14 jaar,V,643.94,646180.62,1157.725563,7.455058e+05
4,OOSTZAAN,10 t/m 14 jaar,V,244.07,211058.87,1157.725563,2.825661e+05


## 3. Create the municipality benchmark

Both relative and absolute measures are retained:

- **SCR** shows proportional deviation from the age–sex-adjusted expectation
- **euro gap** shows the absolute difference between actual and expected spending
- **euro gap per insured year** makes the absolute difference more comparable across municipality sizes

These measures answer different questions and should be interpreted together.

In [4]:
benchmark = (
    df_with_rates
    .groupby(
        "gemeentenaam",
        observed=True,
    )
    .agg(
        insured_years=(
            "aantal_verzekerdejaren",
            "sum",
        ),
        actual_cost=(
            "total_healthcare_cost",
            "sum",
        ),
        expected_cost=(
            "expected_cost",
            "sum",
        ),
    )
    .reset_index()
)

benchmark["actual_cost_per_insured_year"] = (
    benchmark["actual_cost"]
    / benchmark["insured_years"]
)

benchmark["expected_cost_per_insured_year"] = (
    benchmark["expected_cost"]
    / benchmark["insured_years"]
)

benchmark["scr"] = (
    benchmark["actual_cost"]
    / benchmark["expected_cost"]
)

benchmark["percentage_difference"] = (
    100 * (benchmark["scr"] - 1)
)

benchmark["euro_gap"] = (
    benchmark["actual_cost"]
    - benchmark["expected_cost"]
)

benchmark["euro_gap_per_insured_year"] = (
    benchmark["euro_gap"]
    / benchmark["insured_years"]
)

benchmark["absolute_scr_deviation"] = (
    benchmark["scr"]
    .sub(1)
    .abs()
)

benchmark = (
    benchmark
    .sort_values(
        "scr",
        ascending=False,
    )
    .reset_index(drop=True)
)

benchmark.insert(
    0,
    "rank_by_scr",
    np.arange(
        1,
        len(benchmark) + 1,
    ),
)

benchmark["rank_by_positive_euro_gap"] = (
    benchmark["euro_gap"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

national_reconciliation = pd.DataFrame(
    {
        "measure": [
            "National actual cost",
            "National expected cost",
            "Difference",
        ],
        "value_eur": [
            benchmark["actual_cost"].sum(),
            benchmark["expected_cost"].sum(),
            (
                benchmark["actual_cost"].sum()
                - benchmark["expected_cost"].sum()
            ),
        ],
    }
)

display(national_reconciliation)

if not np.isclose(
    benchmark["actual_cost"].sum(),
    benchmark["expected_cost"].sum(),
    rtol=1e-10,
    atol=0.01,
):
    raise AssertionError(
        "National actual and expected costs do not reconcile."
    )

display(
    benchmark[
        [
            "rank_by_scr",
            "gemeentenaam",
            "insured_years",
            "scr",
            "percentage_difference",
            "euro_gap",
            "euro_gap_per_insured_year",
        ]
    ].head(10)
)

,measure,value_eur
0,National actual cost,5.502051e+10
1,National expected cost,5.502051e+10
2,Difference,7.629395e-06


,rank_by_scr,gemeentenaam,insured_years,scr,percentage_difference,euro_gap,euro_gap_per_insured_year
0,1,HEERLEN,85434.42,1.243003,24.300285,6.981817e+07,817.213603
1,2,BRUNSSUM,27259.22,1.219121,21.912067,2.088651e+07,766.218274
2,3,KERKRADE,42638.07,1.210175,21.017479,3.131850e+07,734.519637
3,4,STADSKANAAL,31379.16,1.187281,18.728075,2.047007e+07,652.345925
4,5,EMMEN,107229.08,1.180589,18.058893,6.480834e+07,604.391489
5,6,ALBRANDSWAARD,26075.78,1.179920,17.991956,1.435865e+07,550.650979
6,7,VEENDAM,27261.27,1.177948,17.794786,1.606955e+07,589.464312
7,8,LANDGRAAF,36228.58,1.171818,17.181848,2.177013e+07,600.910335
8,9,PEKELA,12060.29,1.167094,16.709444,6.650240e+06,551.416277
9,10,SITTARD-GELEEN,90763.16,1.148682,14.868175,4.629372e+07,510.049668


## 4. Add a transparent population-size caution flag

This dataset does not provide the information needed to calculate formal statistical confidence intervals for healthcare cost ratios.

Instead, the bottom quartile of municipalities by insured years is marked with a **population-size caution flag**. This is a descriptive sensitivity aid, not a formal confidence classification.

The full municipality ranking is retained. A second ranking view excludes flagged municipalities only to show how sensitive the extremes are to this simple size rule.

In [5]:
population_q1 = (
    benchmark["insured_years"]
    .quantile(0.25)
)

population_median = (
    benchmark["insured_years"]
    .median()
)

benchmark["small_population_caution_flag"] = (
    benchmark["insured_years"]
    < population_q1
)

benchmark["population_size_quartile"] = pd.qcut(
    benchmark["insured_years"],
    q=4,
    labels=[
        "Q1 - smallest",
        "Q2",
        "Q3",
        "Q4 - largest",
    ],
)

pearson_size_association = (
    benchmark["insured_years"]
    .corr(
        benchmark["absolute_scr_deviation"]
    )
)

spearman_size_association = (
    benchmark["insured_years"]
    .rank()
    .corr(
        benchmark["absolute_scr_deviation"]
        .rank()
    )
)

print(
    f"Median municipality size: "
    f"{population_median:,.0f} insured years"
)

print(
    f"Bottom-quartile caution threshold: "
    f"{population_q1:,.0f} insured years"
)

print(
    f"Pearson association between size and absolute SCR deviation: "
    f"{pearson_size_association:.3f}"
)

print(
    f"Spearman rank association between size and absolute SCR deviation: "
    f"{spearman_size_association:.3f}"
)

print(
    "\nThese associations are descriptive. "
    "The caution flag is not a statistical confidence interval."
)

Median municipality size: 32,100 insured years
Bottom-quartile caution threshold: 22,492 insured years
Pearson association between size and absolute SCR deviation: -0.088
Spearman rank association between size and absolute SCR deviation: -0.168

These associations are descriptive. The caution flag is not a statistical confidence interval.


In [6]:
top_10_all = (
    benchmark
    .nlargest(
        10,
        "scr",
    )
)

bottom_10_all = (
    benchmark
    .nsmallest(
        10,
        "scr",
    )
)

benchmark_not_flagged = (
    benchmark.loc[
        ~benchmark[
            "small_population_caution_flag"
        ]
    ]
    .copy()
)

top_10_not_flagged = (
    benchmark_not_flagged
    .nlargest(
        10,
        "scr",
    )
)

bottom_10_not_flagged = (
    benchmark_not_flagged
    .nsmallest(
        10,
        "scr",
    )
)

top_overlap = len(
    set(top_10_all["gemeentenaam"])
    & set(
        top_10_not_flagged["gemeentenaam"]
    )
)

bottom_overlap = len(
    set(bottom_10_all["gemeentenaam"])
    & set(
        bottom_10_not_flagged["gemeentenaam"]
    )
)

sensitivity_summary = pd.DataFrame(
    {
        "comparison": [
            "Top 10 SCR overlap",
            "Bottom 10 SCR overlap",
            "Municipalities flagged by size heuristic",
        ],
        "value": [
            top_overlap,
            bottom_overlap,
            int(
                benchmark[
                    "small_population_caution_flag"
                ].sum()
            ),
        ],
        "denominator_or_context": [
            "out of 10",
            "out of 10",
            f"out of {len(benchmark)}",
        ],
    }
)

display(sensitivity_summary)

comparison_columns = [
    "rank_by_scr",
    "gemeentenaam",
    "insured_years",
    "scr",
    "percentage_difference",
    "euro_gap",
    "small_population_caution_flag",
]

print("Highest SCR values — all municipalities:")
display(top_10_all[comparison_columns])

print("Lowest SCR values — all municipalities:")
display(bottom_10_all[comparison_columns])

print("Highest SCR values — excluding size-flagged municipalities:")
display(top_10_not_flagged[comparison_columns])

print("Lowest SCR values — excluding size-flagged municipalities:")
display(bottom_10_not_flagged[comparison_columns])

,comparison,value,denominator_or_context
0,Top 10 SCR overlap,9,out of 10
1,Bottom 10 SCR overlap,2,out of 10
2,Municipalities flagged by size heuristic,86,out of 342


Highest SCR values — all municipalities:


,rank_by_scr,gemeentenaam,insured_years,scr,percentage_difference,euro_gap,small_population_caution_flag
0,1,HEERLEN,85434.42,1.243003,24.300285,6.981817e+07,False
1,2,BRUNSSUM,27259.22,1.219121,21.912067,2.088651e+07,False
2,3,KERKRADE,42638.07,1.210175,21.017479,3.131850e+07,False
3,4,STADSKANAAL,31379.16,1.187281,18.728075,2.047007e+07,False
4,5,EMMEN,107229.08,1.180589,18.058893,6.480834e+07,False
5,6,ALBRANDSWAARD,26075.78,1.179920,17.991956,1.435865e+07,False
6,7,VEENDAM,27261.27,1.177948,17.794786,1.606955e+07,False
7,8,LANDGRAAF,36228.58,1.171818,17.181848,2.177013e+07,False
8,9,PEKELA,12060.29,1.167094,16.709444,6.650240e+06,True
9,10,SITTARD-GELEEN,90763.16,1.148682,14.868175,4.629372e+07,False


Lowest SCR values — all municipalities:


,rank_by_scr,gemeentenaam,insured_years,scr,percentage_difference,euro_gap,small_population_caution_flag
341,342,TERSCHELLING,4776.29,0.813409,-18.659137,-3.030186e+06,True
340,341,HILVARENBEEK,15835.29,0.815794,-18.420561,-9.793017e+06,True
339,340,DRECHTERLAND,20115.29,0.839327,-16.067312,-1.050307e+07,True
338,339,OUDEWATER,10072.37,0.842776,-15.722394,-5.204618e+06,True
337,338,KOGGENLAND,23176.58,0.847309,-15.269107,-1.139035e+07,False
336,337,TEXEL,13488.22,0.850279,-14.972134,-7.142139e+06,True
335,336,AMELAND,3737.07,0.855534,-14.446550,-1.808410e+06,True
334,335,NIEUWKOOP,29101.91,0.856226,-14.377420,-1.357380e+07,False
333,334,OOSTZAAN,9585.02,0.858974,-14.102612,-4.408404e+06,True
332,333,MIDDEN-DELFLAND,19175.55,0.860904,-13.909557,-8.519955e+06,True


Highest SCR values — excluding size-flagged municipalities:


,rank_by_scr,gemeentenaam,insured_years,scr,percentage_difference,euro_gap,small_population_caution_flag
0,1,HEERLEN,85434.42,1.243003,24.300285,6.981817e+07,False
1,2,BRUNSSUM,27259.22,1.219121,21.912067,2.088651e+07,False
2,3,KERKRADE,42638.07,1.210175,21.017479,3.131850e+07,False
3,4,STADSKANAAL,31379.16,1.187281,18.728075,2.047007e+07,False
4,5,EMMEN,107229.08,1.180589,18.058893,6.480834e+07,False
5,6,ALBRANDSWAARD,26075.78,1.179920,17.991956,1.435865e+07,False
6,7,VEENDAM,27261.27,1.177948,17.794786,1.606955e+07,False
7,8,LANDGRAAF,36228.58,1.171818,17.181848,2.177013e+07,False
9,10,SITTARD-GELEEN,90763.16,1.148682,14.868175,4.629372e+07,False
10,11,MAASTRICHT,110576.28,1.138132,13.813187,5.120574e+07,False


Lowest SCR values — excluding size-flagged municipalities:


,rank_by_scr,gemeentenaam,insured_years,scr,percentage_difference,euro_gap,small_population_caution_flag
337,338,KOGGENLAND,23176.58,0.847309,-15.269107,-1.139035e+07,False
334,335,NIEUWKOOP,29101.91,0.856226,-14.377420,-1.357380e+07,False
329,330,MEDEMBLIK,45175.91,0.866020,-13.398002,-1.962921e+07,False
328,329,BERGEN NH,29355.42,0.867095,-13.290536,-1.521434e+07,False
326,327,AALSMEER,32264.44,0.869842,-13.015793,-1.312033e+07,False
324,325,KAAG EN BRAASSEM,28641.08,0.874306,-12.569427,-1.157925e+07,False
323,324,LANSINGERLAND,64866.63,0.875150,-12.484995,-2.368988e+07,False
322,323,HOLLANDS KROON,48567.34,0.877042,-12.295815,-1.935576e+07,False
320,321,SINT-MICHIELSGESTEL,29881.93,0.882231,-11.776878,-1.150568e+07,False
318,319,SCHAGEN,50792.34,0.887379,-11.262143,-1.937088e+07,False


## 5. Map the detailed Vektis fields to 11 care categories

The 26 source cost columns are grouped into 11 broader care categories to make the results easier to interpret.

The mapping is explicit and saved as an output table so that the aggregation remains transparent.

In [7]:
category_map = {
    "kosten_medisch_specialistische_z": "Medical specialist care",
    "kosten_farmacie": "Pharmacy",
    "kosten_huisarts_inschrijftarief": "General practice",
    "kosten_huisarts_consult": "General practice",
    "kosten_huisarts_mdz": "General practice",
    "kosten_huisarts_overig": "General practice",
    "kosten_hulpmiddelen": "Medical devices",
    "kosten_mondzorg": "Dental care",
    "kosten_paramedische_zorg_fysioth": "Paramedical care",
    "kosten_paramedische_zorg_overig": "Paramedical care",
    "kosten_ziekenvervoer_zittend": "Patient transport",
    "kosten_ziekenvervoer_liggend": "Patient transport",
    "kosten_kraamzorg": "Maternity care",
    "kosten_verloskundige_zorg": "Maternity care",
    "kosten_integrale_geboortezorg": "Maternity care",
    "kosten_grensoverschrijdende_zorg": "Other Zvw care",
    "kosten_eerstelijns_ondersteuning": "Other Zvw care",
    "kosten_gzsp": "Other Zvw care",
    "kosten_overig": "Other Zvw care",
    "kosten_geriatrische_revalidatiez": (
        "Nursing, rehabilitation and short-stay care"
    ),
    "kosten_eerstelijnsverblijf": (
        "Nursing, rehabilitation and short-stay care"
    ),
    "kosten_verpleging_en_verzorging": (
        "Nursing, rehabilitation and short-stay care"
    ),
    "kosten_innovatiegelden_ggz": "Mental health (GGZ)",
    "kosten_consulten_ggz": "Mental health (GGZ)",
    "kosten_intramuraal_verblijf_ggz": "Mental health (GGZ)",
    "kosten_overige_prestaties_ggz": "Mental health (GGZ)",
}

if set(category_map) != set(cost_cols):
    unmapped_columns = set(cost_cols).difference(category_map)
    unexpected_columns = set(category_map).difference(cost_cols)
    raise ValueError(
        "Category mapping does not match the source cost columns. "
        f"Unmapped: {sorted(unmapped_columns)}; "
        f"Unexpected: {sorted(unexpected_columns)}"
    )

category_mapping_table = pd.DataFrame(
    {
        "source_cost_column": list(category_map.keys()),
        "care_category": list(category_map.values()),
    }
).sort_values(
    ["care_category", "source_cost_column"]
)

print(
    "Source cost columns:",
    len(category_map),
)

print(
    "Broader care categories:",
    category_mapping_table[
        "care_category"
    ].nunique(),
)

display(category_mapping_table)

Source cost columns: 26
Broader care categories: 11


,source_cost_column,care_category
7,kosten_mondzorg,Dental care
3,kosten_huisarts_consult,General practice
2,kosten_huisarts_inschrijftarief,General practice
4,kosten_huisarts_mdz,General practice
5,kosten_huisarts_overig,General practice
14,kosten_integrale_geboortezorg,Maternity care
12,kosten_kraamzorg,Maternity care
13,kosten_verloskundige_zorg,Maternity care
6,kosten_hulpmiddelen,Medical devices
0,kosten_medisch_specialistische_z,Medical specialist care


## 6. Calculate category-level actual and expected costs

The same indirect-standardisation method is applied separately to each care category.

This makes it possible to distinguish:

- the category with the largest **relative ratio**
- the category with the largest **absolute euro contribution**

Those are not necessarily the same.

In [8]:
category_long = df.melt(
    id_vars=[
        "geslacht",
        "leeftijdsklasse",
        "gemeentenaam",
        "aantal_verzekerdejaren",
    ],
    value_vars=list(category_map),
    var_name="source_cost_column",
    value_name="actual_cost",
)

category_long["care_category"] = (
    category_long[
        "source_cost_column"
    ]
    .map(category_map)
)

municipality_age_sex_category = (
    category_long
    .groupby(
        [
            "geslacht",
            "leeftijdsklasse",
            "gemeentenaam",
            "care_category",
        ],
        observed=True,
    )
    .agg(
        actual_cost=(
            "actual_cost",
            "sum",
        ),
        insured_years=(
            "aantal_verzekerdejaren",
            "first",
        ),
    )
    .reset_index()
)

national_category_reference = (
    municipality_age_sex_category
    .groupby(
        [
            "geslacht",
            "leeftijdsklasse",
            "care_category",
        ],
        observed=True,
    )
    .agg(
        national_actual_cost=(
            "actual_cost",
            "sum",
        ),
        national_insured_years=(
            "insured_years",
            "sum",
        ),
    )
    .reset_index()
)

national_category_reference[
    "national_rate_per_insured_year"
] = (
    national_category_reference[
        "national_actual_cost"
    ]
    / national_category_reference[
        "national_insured_years"
    ]
)

municipality_age_sex_category = (
    municipality_age_sex_category
    .merge(
        national_category_reference[
            [
                "geslacht",
                "leeftijdsklasse",
                "care_category",
                "national_rate_per_insured_year",
            ]
        ],
        on=[
            "geslacht",
            "leeftijdsklasse",
            "care_category",
        ],
        how="left",
        validate="many_to_one",
    )
)

municipality_age_sex_category[
    "expected_cost"
] = (
    municipality_age_sex_category[
        "insured_years"
    ]
    * municipality_age_sex_category[
        "national_rate_per_insured_year"
    ]
)

category_benchmark = (
    municipality_age_sex_category
    .groupby(
        [
            "gemeentenaam",
            "care_category",
        ],
        observed=True,
    )
    .agg(
        actual_cost=(
            "actual_cost",
            "sum",
        ),
        expected_cost=(
            "expected_cost",
            "sum",
        ),
    )
    .reset_index()
)

category_benchmark = (
    category_benchmark
    .merge(
        benchmark[
            [
                "gemeentenaam",
                "insured_years",
                "actual_cost",
                "expected_cost",
            ]
        ].rename(
            columns={
                "actual_cost": "municipality_actual_cost",
                "expected_cost": "municipality_expected_cost",
            }
        ),
        on="gemeentenaam",
        how="left",
        validate="many_to_one",
    )
)

category_benchmark["ratio"] = (
    category_benchmark["actual_cost"]
    / category_benchmark[
        "expected_cost"
    ].replace(0, np.nan)
)

category_benchmark["percentage_difference"] = (
    100
    * (
        category_benchmark["ratio"]
        - 1
    )
)

category_benchmark["euro_gap"] = (
    category_benchmark["actual_cost"]
    - category_benchmark["expected_cost"]
)

category_benchmark[
    "actual_cost_per_insured_year"
] = (
    category_benchmark["actual_cost"]
    / category_benchmark["insured_years"]
)

category_benchmark[
    "expected_cost_per_insured_year"
] = (
    category_benchmark["expected_cost"]
    / category_benchmark["insured_years"]
)

category_benchmark[
    "actual_cost_share_percent"
] = (
    100
    * category_benchmark["actual_cost"]
    / category_benchmark[
        "municipality_actual_cost"
    ]
)

category_reconciliation = (
    category_benchmark
    .groupby(
        "gemeentenaam",
        observed=True,
    )
    .agg(
        category_actual_cost=(
            "actual_cost",
            "sum",
        ),
        category_expected_cost=(
            "expected_cost",
            "sum",
        ),
    )
    .reset_index()
    .merge(
        benchmark[
            [
                "gemeentenaam",
                "actual_cost",
                "expected_cost",
            ]
        ],
        on="gemeentenaam",
        how="left",
        validate="one_to_one",
    )
)

max_actual_reconciliation_error = (
    category_reconciliation[
        "category_actual_cost"
    ]
    .sub(
        category_reconciliation[
            "actual_cost"
        ]
    )
    .abs()
    .max()
)

max_expected_reconciliation_error = (
    category_reconciliation[
        "category_expected_cost"
    ]
    .sub(
        category_reconciliation[
            "expected_cost"
        ]
    )
    .abs()
    .max()
)

print(
    "Maximum category-to-overall actual-cost reconciliation error:",
    f"EUR {max_actual_reconciliation_error:,.8f}",
)

print(
    "Maximum category-to-overall expected-cost reconciliation error:",
    f"EUR {max_expected_reconciliation_error:,.8f}",
)

if max_actual_reconciliation_error > 0.01:
    raise AssertionError(
        "Category actual costs do not reconcile with the overall benchmark."
    )

if max_expected_reconciliation_error > 0.01:
    raise AssertionError(
        "Category expected costs do not reconcile with the overall benchmark."
    )

display(category_benchmark.head())

Maximum category-to-overall actual-cost reconciliation error: EUR 0.00000024
Maximum category-to-overall expected-cost reconciliation error: EUR 0.00000024


,gemeentenaam,care_category,actual_cost,expected_cost,insured_years,municipality_actual_cost,municipality_expected_cost,ratio,percentage_difference,euro_gap,actual_cost_per_insured_year,expected_cost_per_insured_year,actual_cost_share_percent
0,AA EN HUNZE,Dental care,1080035.22,1.367464e+06,25395.54,82970703.81,9.052228e+07,0.789809,-21.019098,-2.874285e+05,42.528539,53.846611,1.301707
1,AA EN HUNZE,General practice,7760679.93,7.358052e+06,25395.54,82970703.81,9.052228e+07,1.054719,5.471933,4.026277e+05,305.592239,289.737972,9.353518
2,AA EN HUNZE,Maternity care,756008.33,7.192586e+05,25395.54,82970703.81,9.052228e+07,1.051094,5.109389,3.674972e+04,29.769335,28.322241,0.911175
3,AA EN HUNZE,Medical devices,3297918.22,3.312359e+06,25395.54,82970703.81,9.052228e+07,0.995640,-0.435961,-1.444061e+04,129.862103,130.430730,3.974798
4,AA EN HUNZE,Medical specialist care,43716239.03,4.873895e+07,25395.54,82970703.81,9.052228e+07,0.896947,-10.305324,-5.022706e+06,1721.414037,1919.193099,52.688765


## 7. Select municipalities for detailed interpretation

For focused visualisation, the analysis selects:

- the three highest SCR municipalities not marked by the size heuristic
- the three lowest SCR municipalities not marked by the size heuristic

This selection supports interpretation but does not imply that these municipalities are proven outliers in a formal statistical sense.

In [10]:
spotlight_high = (
    benchmark_not_flagged
    .nlargest(
        3,
        "scr",
    )["gemeentenaam"]
    .tolist()
)

spotlight_low = (
    benchmark_not_flagged
    .nsmallest(
        3,
        "scr",
    )["gemeentenaam"]
    .tolist()
)

spotlight_municipalities = (
    spotlight_high
    + spotlight_low
)

print(
    "Higher-than-expected spotlight municipalities:",
    spotlight_high,
)

print(
    "Lower-than-expected spotlight municipalities:",
    spotlight_low,
)

# Rename municipality-level metrics before merging so they remain
# clearly separated from the category-level metrics.
municipality_spotlight_metrics = (
    benchmark[
        [
            "gemeentenaam",
            "rank_by_scr",
            "scr",
            "percentage_difference",
            "euro_gap",
            "small_population_caution_flag",
        ]
    ]
    .rename(
        columns={
            "scr": "municipality_scr",
            "percentage_difference": (
                "municipality_percentage_difference"
            ),
            "euro_gap": "municipality_euro_gap",
        }
    )
)

category_breakdown_spotlight = (
    category_benchmark.loc[
        category_benchmark[
            "gemeentenaam"
        ].isin(
            spotlight_municipalities
        )
    ]
    .merge(
        municipality_spotlight_metrics,
        on="gemeentenaam",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "municipality_scr",
            "euro_gap",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    category_breakdown_spotlight[
        [
            "gemeentenaam",
            "care_category",
            "rank_by_scr",
            "municipality_scr",
            "municipality_percentage_difference",
            "ratio",
            "actual_cost",
            "expected_cost",
            "euro_gap",
            "municipality_euro_gap",
            "actual_cost_share_percent",
            "small_population_caution_flag",
        ]
    ]
)

Higher-than-expected spotlight municipalities: ['HEERLEN', 'BRUNSSUM', 'KERKRADE']
Lower-than-expected spotlight municipalities: ['KOGGENLAND', 'NIEUWKOOP', 'MEDEMBLIK']


,gemeentenaam,care_category,rank_by_scr,municipality_scr,municipality_percentage_difference,ratio,actual_cost,expected_cost,euro_gap,municipality_euro_gap,actual_cost_share_percent,small_population_caution_flag
0,HEERLEN,Medical specialist care,1,1.243003,24.300285,1.139029,1.725047e+08,1.514490e+08,2.105575e+07,6.981817e+07,48.302735,False
1,HEERLEN,Mental health (GGZ),1,1.243003,24.300285,1.784797,4.282290e+07,2.399315e+07,1.882976e+07,6.981817e+07,11.990765,False
2,HEERLEN,"Nursing, rehabilitation and short-stay care",1,1.243003,24.300285,1.306744,3.318573e+07,2.539573e+07,7.789999e+06,6.981817e+07,9.292277,False
3,HEERLEN,Pharmacy,1,1.243003,24.300285,1.215892,3.771734e+07,3.102031e+07,6.697033e+06,6.981817e+07,10.561167,False
4,HEERLEN,General practice,1,1.243003,24.300285,1.186887,2.848911e+07,2.400322e+07,4.485888e+06,6.981817e+07,7.977184,False
...,...,...,...,...,...,...,...,...,...,...,...,...
61,KOGGENLAND,General practice,338,0.847309,-15.269107,0.872442,5.534538e+06,6.343734e+06,-8.091963e+05,-1.139035e+07,8.756207,False
62,KOGGENLAND,Pharmacy,338,0.847309,-15.269107,0.826874,6.641075e+06,8.031540e+06,-1.390465e+06,-1.139035e+07,10.506862,False
63,KOGGENLAND,"Nursing, rehabilitation and short-stay care",338,0.847309,-15.269107,0.688556,4.111364e+06,5.970998e+06,-1.859634e+06,-1.139035e+07,6.504599,False
64,KOGGENLAND,Mental health (GGZ),338,0.847309,-15.269107,0.518177,3.265604e+06,6.302098e+06,-3.036494e+06,-1.139035e+07,5.166520,False


## 8. Save reproducible tables

The saved tables are used by the visualisation notebook and Streamlit app.

The population-size field is deliberately named `small_population_caution_flag` because it is a heuristic warning, not a formal measure of statistical confidence.

In [11]:
OUTPUT_TABLES.mkdir(
    parents=True,
    exist_ok=True,
)

output_files = {
    "municipality_benchmark.csv": benchmark,
    "national_age_sex_reference_rates.csv": (
        age_sex_reference.drop(
            columns="age_sort"
        )
    ),
    "benchmark_sensitivity_summary.csv": sensitivity_summary,
    "category_mapping.csv": category_mapping_table,
    "category_breakdown_all_municipalities.csv": category_benchmark,
    "category_breakdown_outliers.csv": category_breakdown_spotlight,
}

for filename, output_df in output_files.items():
    output_path = OUTPUT_TABLES / filename
    output_df.to_csv(
        output_path,
        index=False,
    )
    print("Saved:", filename)

Saved: municipality_benchmark.csv
Saved: national_age_sex_reference_rates.csv
Saved: benchmark_sensitivity_summary.csv
Saved: category_mapping.csv
Saved: category_breakdown_all_municipalities.csv
Saved: category_breakdown_outliers.csv


## Benchmarking conclusion

This notebook creates an age–sex-adjusted benchmark for all 342 municipalities.

The main analytical lessons are:

- raw municipality costs should not be compared without considering population structure
- SCR provides a relative benchmark, while the euro gap provides an absolute perspective
- the simple population-size caution flag is useful for sensitivity analysis but is not a confidence interval
- ranking results are more sensitive at the low-SCR end, where several smaller municipalities appear among the extremes
- the 26 detailed Vektis cost fields are transparently grouped into **11 care categories**
- category ratios and category euro gaps should be interpreted separately because the highest relative ratio is not always the largest absolute contributor

The results remain descriptive. They identify patterns that may justify further investigation, but they do not establish inefficiency, unnecessary care, or differences in healthcare quality.

Next: `03_results_and_visualization.ipynb` converts the benchmark outputs into clear figures and a concise analytical interpretation.